In [9]:
# =============================================================================
# Zelle 01 – Setup & finale Champion-Modelle (verifiziert per Code)
# =============================================================================
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import train_test_split

SEED = 42
Path("../models").mkdir(exist_ok=True)

# --- Modell B: Ridge, alpha=1.0 (Mehrheitswert, 4/5 Folds, code-verifiziert) ---
MODELL_B_FINAL = Ridge(alpha=1.0, random_state=SEED)

# --- Modell A: LogisticRegression, C=0.1, penalty=l2 (Mehrheitswert, 3/5 bzw. 4/5 Folds, code-verifiziert) ---
MODELL_A_FINAL = LogisticRegression(
    C=0.1, penalty="l2", solver="liblinear", class_weight="balanced",
    max_iter=1000, random_state=SEED
)

print("Modell B (Ridge) Parameter:")
print(MODELL_B_FINAL.get_params())
print("\nModell A (LogisticRegression) Parameter:")
print(MODELL_A_FINAL.get_params())

Modell B (Ridge) Parameter:
{'alpha': 1.0, 'copy_X': True, 'fit_intercept': True, 'max_iter': None, 'positive': False, 'random_state': 42, 'solver': 'auto', 'tol': 0.0001}

Modell A (LogisticRegression) Parameter:
{'C': 0.1, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': 0.0, 'max_iter': 1000, 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'liblinear', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}


In [10]:
# =============================================================================
# Zelle 02 – Modell B (Ridge): Training auf vollem Trainingsset + Speicherung
# =============================================================================
from preprocessing import load_dataset_b, baue_preprocessing_pipeline_b, Y_B_MERKMALE

df_b = load_dataset_b("../data/processed/model_b_preprocessed.csv")

# Identischer Split wie in Notebook 11/12, fuer konsistente Test-Bewertung
train_idx_b, test_idx_b = train_test_split(df_b.index, test_size=0.2, random_state=SEED)

prep_b_final = baue_preprocessing_pipeline_b("original")
X_train_b = prep_b_final.fit_transform(df_b.loc[train_idx_b])
X_test_b = prep_b_final.transform(df_b.loc[test_idx_b])
y_train_b = df_b.loc[train_idx_b, Y_B_MERKMALE]
y_test_b = df_b.loc[test_idx_b, Y_B_MERKMALE]

MODELL_B_FINAL.fit(X_train_b, y_train_b)

from sklearn.metrics import r2_score
y_pred_test_b = MODELL_B_FINAL.predict(X_test_b)
test_r2_b = r2_score(y_test_b, y_pred_test_b)
print(f"Modell B finales Training abgeschlossen. Test-R² (Durchschnitt): {test_r2_b:.4f}")

# --- Speichern: Modell + Preprocessing-Pipeline gemeinsam (fuer konsistente Inferenz) ---
joblib.dump({"modell": MODELL_B_FINAL, "pipeline": prep_b_final, "y_spalten": Y_B_MERKMALE},
            "../models/modell_b_ridge_final.joblib")
print("Gespeichert in: ../models/modell_b_ridge_final.joblib")

Modell B finales Training abgeschlossen. Test-R² (Durchschnitt): 0.5912
Gespeichert in: ../models/modell_b_ridge_final.joblib


In [11]:
# =============================================================================
# Zelle 03 – Modell A (LogisticRegression): Training auf vollem Trainingsset
# =============================================================================
# Datenladen und Split EXAKT wie in Notebook 07 (Zelle 2) reproduziert -
# stratifizierter Split, nicht einfacher train_test_split.
# =============================================================================
from preprocessing import load_dataset
from sklearn.preprocessing import StandardScaler

df_a = load_dataset("../data/processed/model_a_preprocessed.csv")

train_idx_a, test_idx_a = train_test_split(
    df_a.index, test_size=0.2, stratify=df_a["io_nio"], random_state=SEED
)

X_A_MERKMALE = ["schneckendrehzahl", "massedurchsatz", "massetemperatur", "massedruck",
                 "duesenspalt", "abzugsgeschwindigkeit", "kalibrierdruck_mbar",
                 "kuehlwassertemperatur", "mfr_charge", "wandtyp_einwandig", "mechanismus_Vakuum"]

scaler_a_final = StandardScaler()
X_train_a = scaler_a_final.fit_transform(df_a.loc[train_idx_a, X_A_MERKMALE])
X_test_a = scaler_a_final.transform(df_a.loc[test_idx_a, X_A_MERKMALE])
y_train_a = df_a.loc[train_idx_a, "io_nio"]
y_test_a = df_a.loc[test_idx_a, "io_nio"]

MODELL_A_FINAL.fit(X_train_a, y_train_a)

from sklearn.metrics import f1_score
y_pred_test_a = MODELL_A_FINAL.predict(X_test_a)
test_f1_a = f1_score(y_test_a, y_pred_test_a, pos_label="NIO", zero_division=0)
print(f"Modell A finales Training abgeschlossen. Test-F1: {test_f1_a:.4f}")

# KORREKTUR: X_train_a (skaliert) mit abspeichern - SHAP LinearExplainer
# braucht eine repraesentative Referenz-Stichprobe der Trainingsdaten als
# Hintergrund, nicht nur den einzelnen Vorhersagefall (siehe App-Problem:
# ohne echte Referenzdaten kollabieren die SHAP-Werte auf eine winzige
# Skala, alle Balken wirken gleich gross).
joblib.dump({
    "modell": MODELL_A_FINAL, "scaler": scaler_a_final, "feature_namen": X_A_MERKMALE,
    "X_train_skaliert": X_train_a,
}, "../models/modell_a_logreg_final.joblib")
print("Gespeichert in: ../models/modell_a_logreg_final.joblib")

Modell A finales Training abgeschlossen. Test-F1: 0.4151
Gespeichert in: ../models/modell_a_logreg_final.joblib


c:\Users\erikg\miniconda3\envs\extrusion-ml\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


In [12]:
# =============================================================================
# Schneller Test von app_inference.py (in einer Notebook-Zelle oder Skript)
# =============================================================================
import sys
sys.path.append('../src')
from app_inference import lade_modell_b, lade_modell_a, vorhersage_modell_b, vorhersage_modell_a

modell_b_geladen = lade_modell_b("../models/modell_b_ridge_final.joblib")
test_auftrag = {
    "material_mfr": 0.7, "dn_ziel": 150, "wandstaerke_soll": 2.2, "dickentoleranz": 0.15,
    "produktionsgeschwindigkeit_soll": 10, "ovalitaet_anforderung": 0.6, "wandtyp_einwandig": 1
}
ergebnis_b = vorhersage_modell_b(modell_b_geladen, test_auftrag)
print(ergebnis_b)

                 merkmal  empfehlung einheit  r2_konfidenz konfidenz_stufe
0      schneckendrehzahl       44.10   U/min          0.83            hoch
1        massetemperatur      204.66      °C          0.47          mittel
2            duesenspalt        2.32      mm          0.95            hoch
3           vakuumniveau       99.56    mbar          0.37         niedrig
4         innenluftdruck       77.90    mbar          0.52          mittel
5  kuehlwassertemperatur       19.21      °C          0.27         niedrig


In [13]:
# =============================================================================
# Test von vorhersage_modell_a
# =============================================================================
modell_a_geladen = lade_modell_a("../models/modell_a_logreg_final.joblib")

test_einstellung = {
    "schneckendrehzahl": 44.1, "massedurchsatz": 35.0, "massetemperatur": 204.7,
    "massedruck": 145.0, "duesenspalt": 2.3, "abzugsgeschwindigkeit": 1.4,
    "kalibrierdruck_mbar": 140.0, "kuehlwassertemperatur": 19.2, "mfr_charge": 0.7,
    "wandtyp_einwandig": 1, "mechanismus_Vakuum": 0
}
ergebnis_a = vorhersage_modell_a(modell_a_geladen, test_einstellung)
print(ergebnis_a)

{'vorhersage': 'IO', 'nio_wahrscheinlichkeit': np.float64(0.3091)}


In [14]:
import importlib
import app_inference
importlib.reload(app_inference)
from app_inference import lade_modell_b, vorhersage_modell_b

modell_b_geladen = lade_modell_b("../models/modell_b_ridge_final.joblib")
test_auftrag = {
    "material_mfr": 0.7, "dn_ziel": 150, "wandstaerke_soll": 2.2, "dickentoleranz": 0.15,
    "produktionsgeschwindigkeit_soll": 10, "ovalitaet_anforderung": 0.6, "wandtyp_einwandig": 1
}
print(vorhersage_modell_b(modell_b_geladen, test_auftrag))

                 merkmal  empfehlung einheit  r2_konfidenz konfidenz_stufe
0      schneckendrehzahl       44.10   U/min          0.83            hoch
1        massetemperatur      204.66      °C          0.47          mittel
2            duesenspalt        2.32      mm          0.95            hoch
3           vakuumniveau       99.56    mbar          0.37         niedrig
4         innenluftdruck       77.90    mbar          0.52          mittel
5  kuehlwassertemperatur       19.21      °C          0.27         niedrig
